In [4]:
import pandas as pd
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
orders = client.shop.orders

In [5]:
orders.insert_one({"_id": 1004, "customer": {"name": "Hung", "city": "Hue"},
                   "status": "paid", "items": []})
for doc in orders.find({"status": "paid"}, {"customer.name": 1, "status": 1}):
    print(doc)

{'_id': 1004, 'customer': {'name': 'Hung'}, 'status': 'paid'}


In [6]:
pipeline = [
    {"$unwind": "$items"},
    {"$group": {"_id": "$items.sku", "units": {"$sum": "$items.qty"}}},
]
df = pd.DataFrame(orders.aggregate(pipeline))     
print(df)

Empty DataFrame
Columns: []
Index: []


In [7]:
orders.insert_many([
    {"order_id": 1, "customer": "Alice", "city": "Hanoi", "items": [{"sku": "A101", "quantity": 2, "price": 100}]},
    {"order_id": 2, "customer": "Bob", "city": "HCMC", "items": [{"sku": "B201", "quantity": 1, "price": 250}, {"sku": "A101", "quantity": 3, "price": 100}], "status": "completed"},
    {"order_id": 3, "customer": "Charlie", "city": "Hanoi", "sku": "C301", "quantity": 5, "price": 50},
    {"order_id": 4, "customer": "David", "city": "Danang", "items": [{"sku": "D401", "quantity": 4, "price": 80}], "discount": 0.1},
    {"order_id": 5, "customer": "Emma", "city": "HCMC", "items": [{"sku": "A101", "quantity": 1, "price": 100}, {"sku": "E501", "quantity": 2, "price": 150}], "payment": {"method": "card","paid": True}}
])

InsertManyResult([ObjectId('6afda7da18ad3489c536641b'), ObjectId('6afda7da18ad3489c536641c'), ObjectId('6afda7da18ad3489c536641d'), ObjectId('6afda7da18ad3489c536641e'), ObjectId('6afda7da18ad3489c536641f')], acknowledged=True)

In [8]:
sku = "A101"

result = orders.find({"$or": [{"items.sku": sku},{"sku": sku}]})

for order in result:
    print(order)

{'_id': ObjectId('6afda7da18ad3489c536641b'), 'order_id': 1, 'customer': 'Alice', 'city': 'Hanoi', 'items': [{'sku': 'A101', 'quantity': 2, 'price': 100}]}
{'_id': ObjectId('6afda7da18ad3489c536641c'), 'order_id': 2, 'customer': 'Bob', 'city': 'HCMC', 'items': [{'sku': 'B201', 'quantity': 1, 'price': 250}, {'sku': 'A101', 'quantity': 3, 'price': 100}], 'status': 'completed'}
{'_id': ObjectId('6afda7da18ad3489c536641f'), 'order_id': 5, 'customer': 'Emma', 'city': 'HCMC', 'items': [{'sku': 'A101', 'quantity': 1, 'price': 100}, {'sku': 'E501', 'quantity': 2, 'price': 150}], 'payment': {'method': 'card', 'paid': True}}


In [10]:
pipeline = [
    {
        "$addFields": {
            "revenue": {
                "$cond": [
                    {"$isArray": "$items"},
                    {
                        "$sum": {
                            "$map": {
                                "input": "$items",
                                "as": "item",
                                "in": {
                                    "$multiply": [
                                        "$$item.quantity",
                                        "$$item.price"
                                    ]
                                }
                            }
                        }
                    },
                    {
                        "$multiply": [
                            {"$ifNull": ["$quantity", 0]},
                            {"$ifNull": ["$price", 0]}
                        ]
                    }
                ]
            }
        }
    },
{
        "$group": {
            "_id": "$city",
            "total_revenue": {"$sum": "$revenue"}
        }
    },
{
        "$sort": {
            "total_revenue": -1
        }
    }
]

result = orders.aggregate(pipeline)

for row in result:
    print(row)

{'_id': 'HCMC', 'total_revenue': 950}
{'_id': 'Hanoi', 'total_revenue': 450}
{'_id': 'Danang', 'total_revenue': 320}
{'_id': None, 'total_revenue': 0}


In [11]:
import random

orders.drop()

cities = ["Hanoi", "HCMC", "Danang", "Haiphong", "Cantho"]
skus = [f"SKU{i:04d}" for i in range(100)]

data = []

for i in range(10_000):
    data.append({
        "order_id": i,
        "city": random.choice(cities),
        "sku": random.choice(skus),
        "quantity": random.randint(1, 10),
        "price": random.randint(50, 500)
    })

orders.insert_many(data)

InsertManyResult([ObjectId('6afda9df18ad3489c5366420'), ObjectId('6afda9df18ad3489c5366421'), ObjectId('6afda9df18ad3489c5366422'), ObjectId('6afda9df18ad3489c5366423'), ObjectId('6afda9df18ad3489c5366424'), ObjectId('6afda9df18ad3489c5366425'), ObjectId('6afda9df18ad3489c5366426'), ObjectId('6afda9df18ad3489c5366427'), ObjectId('6afda9df18ad3489c5366428'), ObjectId('6afda9df18ad3489c5366429'), ObjectId('6afda9df18ad3489c536642a'), ObjectId('6afda9df18ad3489c536642b'), ObjectId('6afda9df18ad3489c536642c'), ObjectId('6afda9df18ad3489c536642d'), ObjectId('6afda9df18ad3489c536642e'), ObjectId('6afda9df18ad3489c536642f'), ObjectId('6afda9df18ad3489c5366430'), ObjectId('6afda9df18ad3489c5366431'), ObjectId('6afda9df18ad3489c5366432'), ObjectId('6afda9df18ad3489c5366433'), ObjectId('6afda9df18ad3489c5366434'), ObjectId('6afda9df18ad3489c5366435'), ObjectId('6afda9df18ad3489c5366436'), ObjectId('6afda9df18ad3489c5366437'), ObjectId('6afda9df18ad3489c5366438'), ObjectId('6afda9df18ad3489c53664

In [12]:
target_sku = "SKU0001"

query = {"sku": target_sku}

explain_before = orders.find(query).explain()

print(explain_before)

{'explainVersion': '1', 'queryPlanner': {'namespace': 'shop.orders', 'parsedQuery': {'sku': {'$eq': 'SKU0001'}}, 'indexFilterSet': False, 'queryHash': '080C58A0', 'planCacheShapeHash': '080C58A0', 'planCacheKey': '7F8364DB', 'optimizationTimeMillis': 60, 'maxIndexedOrSolutionsReached': False, 'maxIndexedAndSolutionsReached': False, 'maxScansToExplodeReached': False, 'prunedSimilarIndexes': False, 'winningPlan': {'isCached': False, 'stage': 'COLLSCAN', 'filter': {'sku': {'$eq': 'SKU0001'}}, 'nss': 'shop.orders', 'direction': 'forward'}, 'rejectedPlans': []}, 'executionStats': {'executionSuccess': True, 'nReturned': 114, 'executionTimeMillis': 136, 'totalKeysExamined': 0, 'totalDocsExamined': 10000, 'executionStages': {'isCached': False, 'stage': 'COLLSCAN', 'filter': {'sku': {'$eq': 'SKU0001'}}, 'nReturned': 114, 'executionTimeMillisEstimate': 47, 'works': 10001, 'advanced': 114, 'needTime': 9886, 'needYield': 0, 'saveState': 3, 'restoreState': 3, 'isEOF': 1, 'nss': 'shop.orders', 'dire

In [13]:
orders.create_index([("sku", 1)])

explain_after = orders.find(query).explain()

print(explain_after)

{'explainVersion': '1', 'queryPlanner': {'namespace': 'shop.orders', 'parsedQuery': {'sku': {'$eq': 'SKU0001'}}, 'indexFilterSet': False, 'queryHash': '080C58A0', 'planCacheShapeHash': '080C58A0', 'planCacheKey': 'CD1D7373', 'optimizationTimeMillis': 39, 'maxIndexedOrSolutionsReached': False, 'maxIndexedAndSolutionsReached': False, 'maxScansToExplodeReached': False, 'prunedSimilarIndexes': False, 'winningPlan': {'isCached': False, 'stage': 'FETCH', 'nss': 'shop.orders', 'inputStage': {'stage': 'IXSCAN', 'nss': 'shop.orders', 'keyPattern': {'sku': 1}, 'indexName': 'sku_1', 'isMultiKey': False, 'multiKeyPaths': {'sku': []}, 'isUnique': False, 'isSparse': False, 'isPartial': False, 'indexVersion': 2, 'direction': 'forward', 'indexBounds': {'sku': ['["SKU0001", "SKU0001"]']}}}, 'rejectedPlans': []}, 'executionStats': {'executionSuccess': True, 'nReturned': 114, 'executionTimeMillis': 66, 'totalKeysExamined': 114, 'totalDocsExamined': 114, 'executionStages': {'isCached': False, 'stage': 'FE

In [14]:
def extract_stats(explain):
    stats = explain.get("executionStats", {})

    return {"execution_time_ms": stats.get("executionTimeMillis"),
        "documents_examined": stats.get("totalDocsExamined"),
        "documents_returned": stats.get("nReturned"),
        "winning_plan": explain.get("queryPlanner", {}).get("winningPlan", {}).get("stage")
    }

print("BEFORE INDEX:")
print(extract_stats(explain_before))

print("\nAFTER INDEX:")
print(extract_stats(explain_after))

BEFORE INDEX:
{'execution_time_ms': 136, 'documents_examined': 10000, 'documents_returned': 114, 'winning_plan': 'COLLSCAN'}

AFTER INDEX:
{'execution_time_ms': 66, 'documents_examined': 114, 'documents_returned': 114, 'winning_plan': 'FETCH'}
